In [3]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [4]:
# --- 0. HARDWARE DIAGNOSTIC ---
print("--- Hardware Diagnostic ---")
print(f"Python Path: {sys.executable}")
print(f"Torch version: {torch.__version__}")

cuda_available = torch.cuda.is_available()

if cuda_available:
    DEVICE = torch.device("cuda")
    print(f"✅ CUDA IS TRUE! Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version in Torch: {torch.version.cuda}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ CUDA IS FALSE.")
    print("\nREPAIR STEPS FOR JUPYTERLAB DESKTOP:")
    print("1. Run this in a cell: !{sys.executable} -m pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu124 --force-reinstall")
    print("2. Restart Kernel from the top menu.")


--- Hardware Diagnostic ---
Python Path: C:\Users\pudis\AppData\Roaming\jupyterlab-desktop\jlab_server\python.exe
Torch version: 2.7.0.dev20250310+cu124
✅ CUDA IS TRUE! Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA Version in Torch: 12.4


In [5]:

# --- 1. SETTINGS & PATHS ---
DATA_DIR = r"D:\studies\LPU\sem_2\CSE275_OPTIMIZATION_TECHINIQUES _FOR _ML\Traffic-Congestion-Predictor-Using-ML\METR-LA"
BATCH_SIZE = 64  
LEARNING_RATE = 0.001
EPOCHS = 300 
SEQ_LEN = 12 
PRED_LEN = 12

In [6]:
# --- 2. DATA LOADING & PREPROCESSING ---
def load_and_preprocess():
    print("\nLoading METR-LA Dataset...")
    h5_file = os.path.join(DATA_DIR, "METR-LA.h5")
    adj_file = os.path.join(DATA_DIR, "adj_METR-LA.pkl")
    
    if not os.path.exists(h5_file):
        raise FileNotFoundError(f"Data file not found at {h5_file}")
        
    df = pd.read_hdf(h5_file)
    with open(adj_file, 'rb') as f:
        sensor_ids, _, adj_mx = pickle.load(f, encoding='latin1')

    
    data = df.values.astype(np.float32)
    n = len(data)
    
    # Chronological Split
    train_data = data[:int(n*0.7)]
    val_data = data[int(n*0.7):int(n*0.85)]
    test_data = data[int(n*0.85):]
    
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    val_scaled = scaler.transform(val_data)
    test_scaled = scaler.transform(test_data)

    def create_sequences(dataset):
        x, y = [], []
        for i in range(len(dataset) - SEQ_LEN - PRED_LEN):
            x.append(dataset[i : i + SEQ_LEN])
            y.append(dataset[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN])
        return np.array(x).astype(np.float32), np.array(y).astype(np.float32)
    
    return create_sequences(train_scaled), create_sequences(val_scaled), create_sequences(test_scaled), adj_mx, scaler

In [7]:
# --- 3. STGAT MODEL ARCHITECTURE ---
class SpatioTemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_nodes, adj):
        super().__init__()
        # Use register_buffer so it moves with the model to GPU/CPU automatically
        self.register_buffer('adj_matrix', torch.from_numpy(adj).float())
        # Convolutional layers reduce sequence length from 12 -> 10 -> 8
        self.temp1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1))
        self.temp2 = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 1))
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.temp1(x)
        # Spatial interaction using the adjacency matrix
        x = torch.matmul(x, self.adj_matrix) 
        x = self.temp2(x)
        return self.bn(self.relu(x))

In [8]:
class TrafficGNN(nn.Module):
    def __init__(self, num_nodes, adj):
        super().__init__()
        self.block1 = SpatioTemporalBlock(1, 32, num_nodes, adj)
        self.block2 = SpatioTemporalBlock(32, 64, num_nodes, adj)
        # Linear layer expecting 4 remaining time steps (12 - 4 - 4)
        self.out = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * num_nodes, num_nodes * PRED_LEN)
        )

    def forward(self, x):
        x = x.unsqueeze(1) # [B, 1, T, N]
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x).view(-1, PRED_LEN, x.shape[-1])

In [9]:
# --- 4. TRAINING LOOP ---
def run_full_pipeline():
    (train_X, train_Y), (val_X, val_Y), (test_X, test_Y), adj, scaler = load_and_preprocess()
    num_nodes = train_X.shape[2]
    
    use_cuda = torch.cuda.is_available()
    train_loader = DataLoader(TensorDataset(torch.from_numpy(train_X), torch.from_numpy(train_Y)), 
                              batch_size=BATCH_SIZE, shuffle=True, pin_memory=use_cuda)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(val_X), torch.from_numpy(val_Y)), 
                            batch_size=BATCH_SIZE, pin_memory=use_cuda)
    
    model = TrafficGNN(num_nodes, adj).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    counter = 0
    # RTX 3050 Mixed Precision Optimization
    amp_scaler = torch.amp.GradScaler('cuda') if use_cuda else None

    best_rmse = float('inf')
    print(f"\nStarting Optimization on {DEVICE}...")

    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            
            if use_cuda:
                with torch.amp.autocast('cuda'):
                    pred = model(bx); loss = criterion(pred, by)
                amp_scaler.scale(loss).backward(); amp_scaler.step(optimizer); amp_scaler.update()
            else:
                pred = model(bx); loss = criterion(pred, by); loss.backward(); optimizer.step()
            t_loss += loss.item()
        
        # Validation
        model.eval()
        v_rmse = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                pred = model(bx)
                v_rmse += torch.sqrt(F.mse_loss(pred, by)).item()
        
        avg_v_rmse = v_rmse / len(val_loader)
        counter+=1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"{counter} Epoch {epoch+1}/{EPOCHS} | Train Loss: {t_loss/len(train_loader):.5f} | Val RMSE: {avg_v_rmse:.5f}")
        
        # Auto-save best model and scaler
        if avg_v_rmse < best_rmse:
            best_rmse = avg_v_rmse
            torch.save(model.state_dict(), "best_traffic_model.pth")
            with open("scaler.pkl", "wb") as f:
                pickle.dump(scaler, f)
            
    print(f"\nOptimization Complete. Best Val RMSE: {best_rmse:.5f}")


In [10]:
if __name__ == "__main__":
    run_full_pipeline()


Loading METR-LA Dataset...

Starting Optimization on cuda...
1 Epoch 1/300 | Train Loss: 6.40607 | Val RMSE: 1.07249
5 Epoch 5/300 | Train Loss: 0.70875 | Val RMSE: 0.81397
10 Epoch 10/300 | Train Loss: 0.38869 | Val RMSE: 0.66612
15 Epoch 15/300 | Train Loss: 0.30518 | Val RMSE: 0.62874
20 Epoch 20/300 | Train Loss: 0.27689 | Val RMSE: 0.60521
25 Epoch 25/300 | Train Loss: 0.25793 | Val RMSE: 0.59756
30 Epoch 30/300 | Train Loss: 0.30992 | Val RMSE: 0.59602
35 Epoch 35/300 | Train Loss: 0.28910 | Val RMSE: 0.59760
40 Epoch 40/300 | Train Loss: 0.27538 | Val RMSE: 0.58510
45 Epoch 45/300 | Train Loss: 0.26515 | Val RMSE: 0.59410
50 Epoch 50/300 | Train Loss: 0.25903 | Val RMSE: 0.59822
55 Epoch 55/300 | Train Loss: 0.25191 | Val RMSE: 0.58738
60 Epoch 60/300 | Train Loss: 0.25063 | Val RMSE: 0.58377
65 Epoch 65/300 | Train Loss: 0.24187 | Val RMSE: 0.59924
70 Epoch 70/300 | Train Loss: 0.32367 | Val RMSE: 0.58538
75 Epoch 75/300 | Train Loss: 0.23373 | Val RMSE: 0.59067
80 Epoch 80/30